# Tallbehandling


Introduksjon: Python filtyper og Pandas
Hva er Pandas?

Pandas er et bibliotek i Python som brukes til å jobbe med data.

pandas er et sentralt verktøy for dataanalyse {cite}`McKinney2010`.

NumPy brukes for numeriske beregninger {cite}`Harris2020NumPy`.

---

Hva brukes det til?

- lese inn data fra filer (CSV, Excel)
- rydde og strukturere data
- analysere tidsserier (som kraftforbruk)
- gjøre beregninger og statistikk


Den viktigste datastrukturen i pandas er:

DataFrame → en tabell med rader og kolonner (som Excel)

Vi legger først inn data i en egen folder som vi kaller 'data'

og deretter leser vi den i pandas.

Når vi skriver det på denne måten:

df = pd.read_csv("../data/load_data.csv")

../ betyr en mappe opp

| Filtype           | Format  | Beskrivelse                                                                 | Bruk i kraftsystemer                        | Kilde |
|------------------|--------|-----------------------------------------------------------------------------|---------------------------------------------|-------|
| CSV (.csv)       | Tekst   | Tabulære data lagret som tekst med skilletegn (kommaseparert)              | ✅ Måledata fra Statnett / ENTSO-E           | Pandas IO / CSV-standard [1](https://pubmed.ncbi.nlm.nih.gov/37122926/)[2](https://arxiv.org/html/2510.09698v1) |
| Excel (.xlsx)    | Binær   | Regnearkformat med støtte for flere ark og metadata                         | ✅ Rapportering og manuell analyse           | Pandas IO-dokumentasjon [1](https://pubmed.ncbi.nlm.nih.gov/37122926/) |
| JSON (.json)     | Tekst   | Hierarkisk datastruktur (objekter/lister), ofte brukt i API-er             | ✅ Sanntidsdata / datakommunikasjon          | JSON som utvekslingsformat [3](https://energifaktanorge.no/en/interactive-map-quick-downloads/quick-downloads/) |
| Parquet (.parquet)| Binær  | Kolonnebasert lagringsformat optimalisert for ytelse                        | ✅ Store historiske datasett                 | Data storage / performance [4](https://publikasjoner.nve.no/rapport/2026/rapport2026_17.pdf) |
| SQLite (.db/.sqlite) | Binær | Relasjonsdatabase lagret i én fil, støtter SQL-spørringer                   | ✅ Strukturert lagring av tidsserier         | Pandas SQL interface [1](https://pubmed.ncbi.nlm.nih.gov/37122926/) |
| Pickle (.pkl)    | Binær   | Serialisering av Python-objekter til binært format                          | ✅ Midlertidig lagring i Python              | Python standard library [3](https://energifaktanorge.no/en/interactive-map-quick-downloads/quick-downloads/) |


In [1]:
import pandas as pd

df = pd.read_csv("../data/load_data.csv", sep=",")
df

,Time(Local),Production,Consumption
0,01.01.2026 00:00:00 +01:00,"18448,59","16662,95"
1,01.01.2026 01:00:00 +01:00,"18693,66","14953,24"
2,01.01.2026 02:00:00 +01:00,"18639,68","14298,07"
3,01.01.2026 03:00:00 +01:00,"18502,66","13254,81"
4,01.01.2026 04:00:00 +01:00,"18458,45","12678,51"
...,...,...,...
3354,20.05.2026 19:00:00 +02:00,"13799,35","19331,51"
3355,20.05.2026 20:00:00 +02:00,"13857,89","20096,14"
3356,20.05.2026 21:00:00 +02:00,"13768,33","19944,19"
3357,20.05.2026 22:00:00 +02:00,"13646,36","19368,15"


In [2]:
print(df.columns)

Index(['Time(Local)', 'Production', 'Consumption'], dtype='object')


## Hva er en tidsserie?
I kraftsystemer jobber vi ofte med tidsserier:
data som er målt over tid
Eksempel:

last (MW)

produksjon

frekvens

### Viktig: DatetimeIndex

En vanlig index er bare “labels”.
Men DatetimeIndex gir deg tidsforståelse innebygd, som betyr at du kan hente ut tid direkte.

 - df.index.hour
 - df.index.day
 - df.index.month
 - df.index.year

Dette er veldig nyttig når vi jobber med tidsserier

For å analysere tidsserier setter vi tid som indeks:

Selv med parse_dates, kan pandas feile hvis formatet er rart.

In [3]:
df = pd.read_csv("../data/load_data.csv", parse_dates=["Time(Local)"])
#df = df.set_index("Time(Local)")

In [4]:
print(df["Time(Local)"].head())
print(type(df["Time(Local)"].iloc[0]))

0    01.01.2026 00:00:00 +01:00
1    01.01.2026 01:00:00 +01:00
2    01.01.2026 02:00:00 +01:00
3    01.01.2026 03:00:00 +01:00
4    01.01.2026 04:00:00 +01:00
Name: Time(Local), dtype: object
<class 'str'>


In [5]:
print(df.columns)

Index(['Time(Local)', 'Production', 'Consumption'], dtype='object')


In [6]:

df.columns = df.columns.str.strip()

df["Time(Local)"] = pd.to_datetime(
    df["Time(Local)"],
    dayfirst=True,
    utc=True   # ✅ THIS solves your error
)

df = df.set_index("Time(Local)")
df = df.sort_index()

df = df.tz_convert("Europe/Oslo")


In [7]:
print(df.columns)

Index(['Production', 'Consumption'], dtype='object')


In [8]:
print(df.head())

                          Production Consumption
Time(Local)                                     
2026-01-01 00:00:00+01:00   18448,59    16662,95
2026-01-01 01:00:00+01:00   18693,66    14953,24
2026-01-01 02:00:00+01:00   18639,68    14298,07
2026-01-01 03:00:00+01:00   18502,66    13254,81
2026-01-01 04:00:00+01:00   18458,45    12678,51


Første linje i dataene 18448,59 er norsk format: , = desimaltegn

pandas har lest det som tekst

## FIX (konverter til tall)

In [9]:
df["Consumption"] = df["Consumption"].str.replace(",", ".").astype(float)

In [10]:
print(df.dtypes)

Production      object
Consumption    float64
dtype: object


Da kan vi gjøre:

In [11]:
analyse = df['Consumption'].resample("1D").mean()
print(analyse.head())

Time(Local)
2026-01-01 00:00:00+01:00    15464.587083
2026-01-02 00:00:00+01:00    19360.323333
2026-01-03 00:00:00+01:00    23015.297083
2026-01-04 00:00:00+01:00    23660.607500
2026-01-05 00:00:00+01:00    25149.064167
Freq: D, Name: Consumption, dtype: float64


## Motivasjon
Hvorfor dette er viktig i dette kurset
I dette kurset bruker vi pandas til å:
✅ lese ekte kraftsystemdata
✅ analysere lastprofiler
✅ finne mønstre i forbruk
✅ koble data til dynamiske modeller

🎯 Kobling videre
Når dataene er klare, kan vi:

lage en representativ døgnprofil
glatte data (Gaussian-filter)
analysere dynamikk (swing equation)